# Why the Crude Death Rate Lies: Age Composition and Simpson's Paradox

*Demographic Analysis Series · Module 1 (Mortality) · Lesson 1.1b*
*Author: Jason Li | Published: August 30, 2026*

---

In [Lesson 1.1a](demographic-m1l1a-lexis-diagram) we built an honest denominator: person-years. But an honest numerator over an honest denominator can still mislead, and this post shows how, using a real case from Carmichael (2016, Ch. 1). In 1988 Malaysia's crude death rate was **4.93** per 1,000 and Australia's was **7.25**. Read naively, mortality was 47% higher in Australia. Yet Australians lived about six years longer, and Australian death rates were lower at nearly every age. Both statements are correct. The rest of this post explains how.

**What you'll learn**

- The master identity behind every crude rate: $M = \sum_c m(c) \cdot p(c)$, specific rates times composition
- How age structure alone can reverse a mortality ranking (Simpson's paradox)
- How to read an age-specific rate table against the crude rate it produces
- A first look at the fix: re-weighting to a common composition (direct standardization, Lesson 1.2)

**Anchors:** Wachter §2.3; Carmichael Ch. 1, pp. 33–35 (Tables 1.2–1.3).


---

## Part 1: The Puzzle

The crude death rates below are both honest rates in the sense of Lesson 1.1a: real deaths divided by real person-years. The trouble starts when we treat them as measures of *underlying mortality conditions*.


In [ ]:
# ---- 1. The 1988 puzzle: CDR says one thing, life expectancy another ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110

# CDR: Carmichael (2016), Ch. 1, p. 33. Life expectancy: approximate UN estimates for 1985-1990.
puzzle = pd.DataFrame({
    'country': ['Malaysia', 'Australia'],
    'CDR_per_1000': [4.93, 7.25],
    'life_expectancy': [70.3, 76.3],
})
print(puzzle.to_string(index=False))
print(f'\nNaive reading: Australian mortality higher by a factor of {7.25/4.93:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, col, title in zip(axes, ['CDR_per_1000', 'life_expectancy'],
                          ['Crude death rate (per 1,000), 1988', 'Life expectancy at birth (years)']):
    ax.bar(puzzle['country'], puzzle[col], color=['#e0826d', '#6d8ee0'], width=0.5)
    for i, v in enumerate(puzzle[col]):
        ax.text(i, v, f'{v:g}', ha='center', va='bottom', fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.margins(y=0.15)
plt.tight_layout()
plt.show()

The two panels contradict each other. The CDR says Australians died at 1.47 times the Malaysian rate; the life expectancies say the opposite. To see why, we need to look inside the crude rate.


---

## Part 2: The Master Identity Behind Every Crude Rate

Every death in 1988 Malaysia or Australia occurred to a person of some sex at some age. Grouping deaths by sex and age group $(s, x)$:

$$
\text{CDR} = \frac{\sum_{s,x} D(s,x)}{P} = \sum_{s,x} \underbrace{\frac{D(s,x)}{P(s,x)}}_{\text{specific rate } m(s,x)} \cdot \underbrace{\frac{P(s,x)}{P}}_{\text{composition } p(s,x)}
$$

A crude rate has exactly two levers: the **rate schedule** $m(s,x)$ (how dangerous each age is) and the **composition** $p(s,x)$ (how many people sit at each age). Carmichael calls this the master identity (his Eq. 2.3); it applies to every ratio-type measure in this series, not just the CDR.

Below we rebuild the 1988 comparison from Carmichael's Tables 1.2 (age-sex-specific death rates) and 1.3 (age-sex composition per million population), and check that we reproduce his published death counts and CDRs.


In [ ]:
# ---- 2. Rebuild the comparison from Carmichael Tables 1.2 and 1.3 ----
from io import StringIO

# age-sex-specific death rates per 1,000 (Table 1.2)
rates_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,16.6,9.8,12.8,7.6
1-4,1.7,0.5,1.1,0.4
5-9,0.6,0.2,0.5,0.2
10-14,0.6,0.3,0.4,0.2
15-19,1.0,1.1,0.5,0.4
20-24,1.5,1.6,0.6,0.5
25-29,1.7,1.5,0.9,0.5
30-34,2.0,1.4,1.1,0.6
35-39,2.4,1.5,1.5,0.8
40-44,3.4,2.2,2.3,1.2
45-49,5.5,3.4,3.2,2.1
50-54,9.9,6.0,5.9,3.4
55-59,15.4,10.0,9.9,5.5
60-64,25.5,17.3,17.0,8.7
65-69,38.3,27.2,27.8,13.8
70-74,62.7,45.3,46.6,23.5
75-79,86.2,71.9,69.8,40.7
80-84,144.3,110.7,119.0,71.4
85+,173.4,186.6,142.8,147.7
"""

# persons per 1,000,000 total population (Table 1.3)
pop_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,14254,7597,13464,7245
1-4,56773,30324,53804,28954
5-9,61207,37762,58012,35803
10-14,56002,38712,53613,36758
15-19,52776,43626,50772,41743
20-24,50193,40771,49060,39297
25-29,42748,42914,44839,41982
30-34,35704,40234,38326,39933
35-39,30427,38778,31269,38391
40-44,23858,36248,23075,34537
45-49,20473,27923,19767,26357
50-54,17052,23855,17190,22798
55-59,12615,22708,13604,21924
60-64,10043,21764,10513,22336
65-69,7373,17701,8498,19973
70-74,4608,12857,5246,16200
75-79,3430,8712,3976,12555
80-84,1399,4471,1675,7827
85+,963,2319,1399,6111
"""

rates = pd.read_csv(StringIO(rates_csv), index_col='age')
pop = pd.read_csv(StringIO(pop_csv), index_col='age')

# deaths in each sex-age group = persons x rate / 1,000
deaths = pop * rates / 1000

# checkpoint: do we reproduce the textbook's published totals?
published_deaths = {'m_malaysia': 2799, 'm_australia': 3939,
                    'f_malaysia': 2152, 'f_australia': 3313}
for col in deaths.columns:
    print(f'{col:12s}: computed {deaths[col].sum():6.0f}  vs  published {published_deaths[col]}')

cdr_malaysia = (deaths['m_malaysia'].sum() + deaths['f_malaysia'].sum()) / 1e6 * 1000
cdr_australia = (deaths['m_australia'].sum() + deaths['f_australia'].sum()) / 1e6 * 1000
print(f'\nComputed CDRs: Malaysia {cdr_malaysia:.2f}, Australia {cdr_australia:.2f} per 1,000')
print('(Textbook: 4.93 and 7.25; small gaps are rounding in the per-million table)')

The numbers check out: our rebuilt death counts match the textbook's published totals, and the two CDRs fall out of the master identity.

Now read the rate columns of Table 1.2 against the CDRs. Australia's death rate is lower than Malaysia's in **almost every sex-age group**, often by a wide margin. The exceptions:

- **Males 15–19 and 20–24**: Australia slightly higher, reflecting fatal road accidents among young men.
- **Both sexes 85+**: Australia higher, but for a compositional reason *inside* the group: proportionately more Australians in the 85+ band are 90+ or 95+, where the risk of dying is highest.

So the age-specific table says the reverse of what the CDRs said. The next figure shows all three pieces of the puzzle side by side: composition, rate schedule, and where the deaths actually land.


In [ ]:
# ---- 3. Three views: composition, rate schedules, and where deaths land ----
plt.rcParams['font.size'] = 13  # figure is wide; keep text legible when scaled into the page column
both_pop = pop['m_malaysia'] + pop['f_malaysia']
aus_pop = pop['m_australia'] + pop['f_australia']
both_deaths = deaths['m_malaysia'] + deaths['f_malaysia']
aus_deaths = deaths['m_australia'] + deaths['f_australia']

x = np.arange(len(rates))
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) composition: % of population in each age group
axes[0].plot(x, both_pop / 1e4, 'o-', color='#e0826d', lw=2, label='Malaysia')
axes[0].plot(x, aus_pop / 1e4, 's-', color='#6d8ee0', lw=2, label='Australia')
axes[0].set_title('(a) Composition: % of population')
axes[0].set_ylabel('% of total population')
axes[0].legend()

# (b) rate schedules (log scale): 4 lines
axes[1].semilogy(x, rates['m_malaysia'], 'o-', color='#e0826d', lw=1.5, label='Malaysia, male')
axes[1].semilogy(x, rates['f_malaysia'], 'o--', color='#b3553f', lw=1.5, label='Malaysia, female')
axes[1].semilogy(x, rates['m_australia'], 's-', color='#6d8ee0', lw=1.5, label='Australia, male')
axes[1].semilogy(x, rates['f_australia'], 's--', color='#3f5cb3', lw=1.5, label='Australia, female')
axes[1].set_title('(b) Rate schedules: deaths per 1,000 (log)')
axes[1].set_ylabel('death rate (log scale)')
axes[1].legend(fontsize=9, ncol=2)

# (c) contribution to the CDR: deaths per 1,000 by age group
w = 0.4
axes[2].bar(x - w/2, both_deaths / 1e3, width=w, color='#e0826d', label='Malaysia')
axes[2].bar(x + w/2, aus_deaths / 1e3, width=w, color='#6d8ee0', label='Australia')
axes[2].set_title('(c) CDR contributions: deaths per 1,000')
axes[2].set_ylabel('deaths per 1,000 population')
axes[2].legend()

for ax in axes:
    ax.set_xticks(x[::2])
    ax.set_xticklabels(rates.index[::2], rotation=45)
    ax.set_xlabel('age group')

plt.tight_layout()
plt.show()

share_65plus_mal = both_pop.loc[['65-69','70-74','75-79','80-84','85+']].sum() / 1e4
share_65plus_aus = aus_pop.loc[['65-69','70-74','75-79','80-84','85+']].sum() / 1e4
print(f'Population aged 65+: Malaysia {share_65plus_mal:.1f}%, Australia {share_65plus_aus:.1f}%')

Read the three panels together:

- **(a)** Malaysia's population in 1988 was concentrated at young ages, a legacy of higher fertility; Australia's had a large mass at older ages (10.9% vs 3.9% aged 65+).
- **(b)** Australia's schedule sits below Malaysia's at nearly every point. Death is cheap at young ages and expensive at old ages, for both countries.
- **(c)** Australia's deaths land overwhelmingly in the old-age groups, where its large elderly population faces rates of 30–180 per 1,000. Malaysia's young composition keeps most of its people in the cheap age groups, and its CDR stays low despite a worse schedule.

Higher fertility gave Malaysia the younger structure, and the composition advantage was large enough to more than offset Australia's better schedule. The CDR comparison is true as a description of mortality *burden* (Australia really did record more deaths per 1,000 people), and false as a comparison of mortality *risk*.


---

## Part 3: Simpson's Paradox, and the Fix

A comparison that reverses when you aggregate is a case of **Simpson's paradox**: within nearly every stratum Australia's mortality was lower, yet the aggregated CDR ranks Australia worse. The mechanism is always the same: a compositional variable (here age) that is related both to the rate and to the group label.

A useful discipline from Carmichael: a compositional variable can distort a comparison only if **both** of these hold:

1. the phenomenon's intensity varies across the variable's categories (mortality varies enormously by age: ✓)
2. the populations being compared are distributed differently across those categories (3.9% vs 10.9% elderly: ✓)

Both conditions hold for age, so the comparison is contaminated. (For sex, condition 2 fails: the two populations are each about half male, so sex composition barely matters here. See Exercise 3.)

What if we remove lever two and hold composition fixed? Apply each country's rate schedule to the *same* population structure:


In [ ]:
# ---- 4. Hold composition fixed: each schedule on each composition ----
# sex-specific rates applied to sex-specific populations
mal_on_aus = (pop['m_australia'] * rates['m_malaysia'] / 1000
              + pop['f_australia'] * rates['f_malaysia'] / 1000).sum() / 1e6 * 1000
aus_on_mal = (pop['m_malaysia'] * rates['m_australia'] / 1000
              + pop['f_malaysia'] * rates['f_australia'] / 1000).sum() / 1e6 * 1000

print('CDRs actually observed (each schedule on its own composition):')
print(f'  Malaysia {cdr_malaysia:.2f}  vs  Australia {cdr_australia:.2f}  per 1,000')
print(f'  -> naive ratio {cdr_australia/cdr_malaysia:.2f}: Australia looks worse\n')
print('With composition held fixed:')
print(f'  Malaysian schedule on Australian composition: {mal_on_aus:.2f} per 1,000')
print(f'  Australian schedule on Malaysian composition: {aus_on_mal:.2f} per 1,000')
print(f'  -> standardized ratio {mal_on_aus/cdr_australia:.2f}: Malaysia is worse')
print(f'  -> cross-check the other way: {cdr_malaysia/aus_on_mal:.2f}: Malaysia is worse')
print('\n(Carmichael Ch. 2 obtains 1.46 and 1.52 from unrounded data; direction inverted either way)')

With composition fixed, the ranking **inverts**: Malaysia's mortality was roughly 45% higher, not 32% lower. The naive ratio and the fixed-composition ratio sit on opposite sides of 1.0, which is Carmichael's diagnostic for composition *inverting* a differential rather than merely muting it.

This re-weighting has a name: **direct standardization**, the subject of Lesson 1.2. There we will formalize it, meet the standard populations demographers actually use (Segi, WHO World Standard), and see why the absolute value of a standardized rate is meaningless while its ratio to another standardized rate is not. Lesson 1.3 covers **indirect standardization** and the SMR, for populations too small to have stable age-specific rates of their own.


---

## Part 4: Takeaways and Exercises

**Takeaways**

1. A crude rate mixes a rate schedule with a composition: $M = \sum_c m(c)\,p(c)$. This master identity applies to every ratio-type measure in the series.
2. Comparing crude rates confounds the two levers. Age is the usual contaminant for mortality, but never the only candidate (sex, marital status, education can all play the role).
3. Simpson's paradox is not a curiosity; in demography it is the default risk whenever populations have different structures. Standardize or decompose **before** ranking populations.
4. A compositional variable confounds only when intensity varies across its categories **and** the compared populations are distributed differently across them. Checking both conditions tells you which variables are worth standardizing for.

**Exercises**

1. Verify one cell of Table 1.3 by hand: Malaysian males aged 60–64 had 10,043 persons per million and a death rate of 25.5 per 1,000. How many deaths does the master identity predict, and does it match the table's 256?
2. Using the DataFrames above, compute the CDR that Malaysia's 1988 population would have had under Australia's rate schedule (you should get about 3.2 per 1,000). Malaysia's actual CDR was 4.93. What does the remaining gap represent?
3. Sex is also related to mortality. Apply the two-condition test to sex composition in this comparison and explain why standardizing for sex alone would change little here.
4. Find the two sex-age groups besides 85+ where Australia's rate exceeds Malaysia's. What explanation does Carmichael give, and what does it suggest about how finely compositional variables should be cut?

---

**Next post: Lesson 1.1c, The Indicator System for Mortality.** We leave composition aside for one post and build the working vocabulary of mortality measurement: age-specific death rates and their log-scale curves, plus IMR, U5MR, and the maternal mortality ratio, each with its own denominator trap.
